# 07. Heldout: лучший режим каждой модели (гипотеза H4)

Ноутбук прогоняет синтетический набор `heldout` (80 описаний) только в лучшем режиме каждой модели по `real_val` (отчёт 07, таблица 7). Модели идут по очереди в одной сессии: загрузка, 80 карточек, выгрузка из памяти. Код загрузки, промпта и метрик скопирован из `07_run_model.ipynb` без изменений, поэтому условия те же, что на `real_val`.

GigaChat 3.1 здесь нет: она квантуется только в `transformers<5`, её heldout запускается в `07_run_model_gigachat.ipynb` с `SPLITS = ["heldout"]` и `CONFIGS = [("few-shot", True)]`.

Как запускать (Colab, A100):
1. В `vkr1_data` на Google Drive должен лежать `heldout.jsonl` (и остальные файлы, как для `07_run_model.ipynb`).
2. Секреты Colab: `HF_TOKEN` (для Llama и Gemma), по желанию `TG_TOKEN` и `TG_CHAT`.
3. Сначала `N = 5` (проверка), потом `N = None`. «Выполнить все»: при первой установке сеанс перезапустится сам, тогда ещё раз «Выполнить все». Если оборвалось — запусти ещё раз: готовые карточки пропускаются, готовые модели тоже.

По оценке из пилота около 1 часа генерации на семь моделей. В конце ячейка H4 считает разрыв `heldout − real_val`.


In [ ]:
# Установка один раз за сессию. Если библиотек нет: ставим и сами перезапускаем сеанс
# (иначе torch подхватывается наполовину старым и падает с «partially initialized module»).
# После перезапуска снова «Выполнить все» — эта ячейка увидит, что всё стоит, и пропустит установку.
# flash-linear-attention нужен Qwen3.5: без него она считает в 5–10 раз медленнее.
import importlib.util, os
need = [p for p in ("xgrammar", "fla", "bitsandbytes", "accelerate") if importlib.util.find_spec(p) is None]
if need:
    print("ставлю:", need)
    %pip -q install -U accelerate bitsandbytes xgrammar flash-linear-attention
    print("перезапуск сеанса… потом снова «Выполнить все»")
    os.kill(os.getpid(), 9)
else:
    print("библиотеки уже стоят, установка пропущена")


In [ ]:
!pip list

## Настройки
Модели и их лучшие режимы — в `BEST`; порядок запуска — `RUN`.

In [ ]:
import json, re, time, random, os, gc, shutil
import numpy as np
import pandas as pd
import requests

MODELS = {
    "qwen35_9b":       "Qwen/Qwen3.5-9B",
    "gemma4_e4b":      "google/gemma-4-E4B-it",
    "llama31_8b":      "meta-llama/Llama-3.1-8B-Instruct",
    "ministral3_8b":   "mistralai/Ministral-3-8B-Instruct-2512-BF16",
    "phi4_mini":       "microsoft/Phi-4-mini-instruct",
    "tlite21_8b":      "t-tech/T-lite-it-2.1",
    "yandex5_lite_8b": "yandex/YandexGPT-5-Lite-8B-instruct",
}

# лучший запуск каждой модели на real_val (отчёт 07, таблица 7): (режим, JSON-схема)
BEST = {
    "gemma4_e4b":      ("few-shot", True),     # 0,938 — лучшая модель
    "yandex5_lite_8b": ("few-shot", False),    # 0,909
    "ministral3_8b":   ("zero-shot", True),    # 0,902
    "qwen35_9b":       ("few-shot", False),    # 0,896 (со схемой столько же)
    "llama31_8b":      ("few-shot", False),    # 0,896
    "tlite21_8b":      ("few-shot", False),    # 0,895 (со схемой столько же)
    "phi4_mini":       ("few-shot", False),    # 0,834 — самая слабая, нужна для H9
    "gigachat31_10b":  ("few-shot", True),     # 0,880 — прогон в 07_run_heldout_gigachat.ipynb (transformers<5)
}

RUN = [m for m in BEST if m in MODELS]     # GigaChat здесь не запускается, но входит в таблицу H4
RUN = ["gemma4_e4b", "phi4_mini"] + [m for m in RUN if m not in ("gemma4_e4b", "phi4_mini")]
SPLITS = ["heldout"]        # real_test не трогаем до этапа 09
AUTO_OFF = True             # отключить среду Colab в конце и при ошибке
N = None                    # сколько карточек взять; None = все
K_SHOT = 5

DATA = "data"
OUT = "results"
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]

try:                        # в Colab данные и результаты лежат на Google Drive, токены — в секретах
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    DATA, OUT = "/content/drive/MyDrive/vkr1_data", "/content/drive/MyDrive/vkr1_results"
    def secret(k):
        try:
            return userdata.get(k)
        except Exception:
            return None
except ImportError:
    secret = os.environ.get
os.makedirs(OUT, exist_ok=True)
HF_TOKEN, TG_BOT_TOKEN, TG_CHAT_ID = secret("HF_TOKEN"), secret("TG_TOKEN"), secret("TG_CHAT")
print("HF:", bool(HF_TOKEN), "| Telegram:", bool(TG_BOT_TOKEN and TG_CHAT_ID))

def tg(text):
    print(text)
    if not (TG_BOT_TOKEN and TG_CHAT_ID):
        return
    try:
        requests.post(f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage", data={"chat_id": TG_CHAT_ID, "text": text}, timeout=10)
    except Exception as e:
        print("telegram не отправился:", type(e).__name__)      # без текста ошибки: в нём был бы токен


## Данные
Карточка = описание и эталон по семи атрибутам (`None`, если в тексте значения нет).
Для реальных карточек эталон берётся из `gold_pairs.csv`, для `heldout` он записан в самом файле.

In [ ]:
cards = {c["url"]: c["description"] for c in json.load(open(f"{DATA}/dns_cards_500.json", encoding="utf-8"))}
pairs = pd.read_csv(f"{DATA}/gold_pairs.csv", dtype=str)
gold_real = {}
for url, g in pairs.groupby("url"):
    gold_real[url] = {a: None for a in ATTRS}          # если атрибута нет в таблице, эталон null
    for _, r in g.iterrows():
        if pd.notna(r["эталон"]):
            gold_real[url][r["атрибут"]] = r["эталон"]
split = json.load(open(f"{DATA}/split_ids.json"))
fewshot = [json.loads(l) for l in open(f"{DATA}/fewshot.jsonl", encoding="utf-8")]
shots = random.Random(0).sample(fewshot, K_SHOT)      # одни и те же примеры для всех моделей

def get_items(name):
    if name == "heldout":
        syn = [json.loads(l) for l in open(f"{DATA}/heldout.jsonl", encoding="utf-8")]
        items = [{"id": f"heldout{i}", "text": r["description"], "gold": r["gold"]} for i, r in enumerate(syn)]
    else:
        items = [{"id": u, "text": cards[u], "gold": gold_real[u]} for u in split[name]]
    return items[:N] if N else items

ITEMS = {s: get_items(s) for s in SPLITS}
print({s: len(v) for s, v in ITEMS.items()})

## Промпт и схема ответа
Промпт одинаковый для всех моделей: одно сообщение пользователя, без system-роли, потому что шаблоны моделей с ней работают по-разному.
В few-shot к инструкции добавляются примеры из `fewshot.jsonl`.

In [ ]:
INSTRUCTION = """Извлеки из описания монитора значения семи характеристик и верни JSON.

Правила:
- Бери только то, что прямо написано в тексте описания. Если характеристики в тексте нет, пиши null.
- Не угадывай и не используй свои знания о модели монитора.
- Все значения строки в таком формате: диагональ — число в дюймах ("27"); разрешение — "2560x1440"; частота — число в герцах ("165"); тип матрицы — "IPS", "VA", "TN" или "OLED"; яркость — число в кд/м² ("300"); контрастность — число ("3000", а 1 млн к 1 пиши "1000000"); изогнутость — "да" или "нет".
- Ответ только JSON с ключами: диагональ, разрешение, частота, тип матрицы, яркость, контрастность, изогнутость."""

def make_prompt(text, mode):
    p = INSTRUCTION
    if mode == "few-shot":
        for i, ex in enumerate(shots, 1):
            p += f"\n\nПример {i}.\nОписание: {ex['description']}\nОтвет: {json.dumps(ex['gold'], ensure_ascii=False)}"
    return p + f"\n\nОписание: {text}\nОтвет:"

SCHEMA_JSON = {"type": "object",
               "properties": {a: {"type": ["string", "null"]} for a in ATTRS},
               "required": ATTRS, "additionalProperties": False}

## Загрузка модели
Та же загрузка, что в `07_run_model.ipynb`, но в функции: модели грузятся по очереди и выгружаются после своего запуска.

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)

In [ ]:
%pip -q install mistral-common

In [ ]:
# GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import xgrammar as xgr
from xgrammar.contrib.hf import LogitsProcessor

tok = model = compiled = None

def load(key):
    """загрузка как в 07_run_model.ipynb: nf4, вычисления в fp16"""
    global tok, model, compiled
    name = MODELS[key]
    kw = {"fix_mistral_regex": True} if key == "ministral3_8b" else {}     # у Ministral иначе неверная токенизация
    tok = AutoTokenizer.from_pretrained(name, **kw)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
    except ValueError:                      # мультимодальные модели иногда грузятся другим классом
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(name, quantization_config=bnb, device_map="auto")
    model.eval()
    print(name, "| память после загрузки, ГБ:", round(torch.cuda.memory_allocated() / 1e9, 1))
    tok_info = xgr.TokenizerInfo.from_huggingface(tok, vocab_size=model.get_output_embeddings().out_features)
    compiled = xgr.GrammarCompiler(tok_info).compile_json_schema(json.dumps(SCHEMA_JSON), any_whitespace=False)

def unload():
    """выгрузить модель и удалить её веса с диска, чтобы хватило места на следующую"""
    global tok, model, compiled
    tok = model = compiled = None
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

def generate(prompt, use_schema):
    """возвращает (ответ модели, секунд, сколько токенов сгенерировано, длина промпта в токенах)"""
    chat = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    processors = [LogitsProcessor(compiled)] if use_schema else []     # для каждого запроса новый
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=200, do_sample=False, logits_processor=processors, stop_strings=["}"], tokenizer=tok)
    sec = time.time() - t0
    new = out[0][x["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), sec, len(new), x["input_ids"].shape[1]

def reset_peak():
    torch.cuda.reset_peak_memory_stats()      # пик памяти считаем отдельно для каждого запуска

## Метрики
Пара «атрибут–значение» считается верной, если значение совпало с эталоном. Если эталон `null`, а модель назвала значение, это ложное значение.
Считаем два варианта: `строго` (текст ответа как есть) и `с нормализацией` (значение приводится правилами из `03_eda`: «27 дюймов» → «27»). Разница между ними показывает, сколько ошибок даёт запись единиц (гипотеза H5).

In [ ]:
def parse(raw):
    """словарь по семи атрибутам или None, если ответ не разобрался как JSON"""
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        d = json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        d = None
    if not isinstance(d, dict):
        return None
    return {a: (None if d.get(a) in (None, "", "null") else str(d[a]).strip()) for a in ATTRS}

def number(s):
    return re.search(r"\d+(?:[.,]\d+)?", s).group().replace(",", ".")

def resolution(s):
    w, h = re.findall(r"\d+", s)[:2]
    return f"{w}x{h}" if int(w) >= 640 else None

def contrast(s):
    s = re.sub(r"(?<=\d)\s(?=\d{3}\b)", "", s.lower())
    v = float(number(s)) * (10**6 if re.search(r"m|м", s) else 1)
    return str(int(v)) if v >= 100 else None

NORM = {"диагональ": lambda s: f"{float(number(s)):g}", "разрешение": resolution, "частота": number,
        "тип матрицы": lambda s: s.upper().replace("QD-", ""), "яркость": number, "контрастность": contrast,
        "изогнутость": lambda s: s.lower()}

def norm(attr, v):
    try:
        return NORM[attr](v)
    except Exception:
        return v

def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0

def score(rows, soft):
    cnt = {a: [0, 0, 0] for a in ATTRS}          # tp, fp, fn по атрибутам
    false_null = []                              # для пар с эталоном null: 1, если модель назвала значение
    exact = []                                   # все семь атрибутов верны
    for r in rows:
        pred = r["pred"] or {a: None for a in ATTRS}
        ok = True
        for a in ATTRS:
            g, p = r["gold"][a], pred[a]
            if soft and p is not None:
                p = norm(a, p)
            if g is None:
                false_null.append(p is not None)
                if p is not None:
                    cnt[a][1] += 1; ok = False
            elif p == g:
                cnt[a][0] += 1
            else:
                cnt[a][2] += 1; ok = False
                if p is not None:
                    cnt[a][1] += 1
        exact.append(ok)
    tp, fp, fn = (sum(c[i] for c in cnt.values()) for i in range(3))
    macro = np.mean([f1(*c) for c in cnt.values() if sum(c)])
    return {"micro_f1": f1(tp, fp, fn), "macro_f1": macro, "exact": np.mean(exact),
            "false_null": np.mean(false_null), "per_attr": {a: round(f1(*c), 3) for a, c in cnt.items()}}

def make_summary(rows, split, mode, schema):
    strict, soft = score(rows, False), score(rows, True)
    sec = [r["sec"] for r in rows]
    summary = {"split": split, "model": MODEL, "mode": mode, "schema": schema, "n": len(rows),
               "hf_id": MODELS[MODEL], "k_shot": K_SHOT if mode == "few-shot" else 0, "дата": time.strftime("%Y-%m-%d %H:%M"),
               "micro_f1_строго": round(strict["micro_f1"], 3), "micro_f1_норм": round(soft["micro_f1"], 3),
               "macro_f1_норм": round(soft["macro_f1"], 3), "exact_норм": round(soft["exact"], 3),
               "ложные_на_null": round(soft["false_null"], 3),
               "невалидных": round(np.mean([r["pred"] is None for r in rows]), 3),
               "сек_среднее": round(np.mean(sec), 2), "сек_p95": round(np.percentile(sec, 95), 2),
               "токенов_среднее": round(np.mean([r["tokens"] for r in rows]), 1),
               "токенов_вход_среднее": round(np.mean([r["tokens_in"] for r in rows if "tokens_in" in r] or [np.nan]), 1)}
    try:
        import torch, transformers
        summary["память_пик_ГБ"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
        summary["transformers"] = transformers.__version__
        summary["gpu"] = torch.cuda.get_device_name(0)       # время и память сравнимы только на одной и той же карте
        from importlib.metadata import version, PackageNotFoundError
        for pkg in ("xgrammar", "flash-linear-attention", "bitsandbytes"):
            try:
                summary[pkg.replace("-", "_")] = version(pkg)       # версии библиотек для журнала параметров
            except PackageNotFoundError:
                summary[pkg.replace("-", "_")] = "нет"
    except Exception:
        pass
    return summary

def save_summary(summary):
    """строка в общий файл (старая строка с тем же набором, моделью и режимом заменяется)"""
    sp = f"{OUT}/summary.csv"
    old = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
    if len(old):
        same = ((old["split"] == summary["split"]) & (old["model"] == summary["model"])
                & (old["mode"] == summary["mode"]) & (old["schema"] == summary["schema"]))
        old = old[~same]
    pd.concat([old, pd.DataFrame([summary])]).to_csv(sp, index=False)

## Прогон
Для каждой модели один запуск — её лучший режим. Готовые карточки и готовые модели пропускаются.

In [ ]:
from tqdm import tqdm
import transformers
transformers.logging.set_verbosity_error()

def run_one(sp_name, mode, schema):
    items = ITEMS[sp_name]
    tag = f"{sp_name}__{MODEL}__{mode}__{'schema' if schema else 'free'}"
    path = f"{OUT}/{tag}.jsonl"
    try:
        rows = list({r["id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}.values())     # дубли по id убираем
    except FileNotFoundError:
        rows = []
    have = {r["id"] for r in rows}
    if len(have) >= len(items):
        print("уже готово:", tag)
        return None
    if model is None:
        load(MODEL)
    print("\n==", tag)
    reset_peak()
    bar = tqdm(total=len(items), initial=len(have), desc=tag)
    for it in items:
        if it["id"] in have:
            continue
        raw, sec, ntok, nin = generate(make_prompt(it["text"], mode), schema)
        row = {"id": it["id"], "gold": it["gold"], "raw": raw, "pred": parse(raw), "sec": round(sec, 2), "tokens": ntok, "tokens_in": nin}
        rows.append(row)
        for attempt in range(5):       # Google Drive иногда на пару секунд теряет папку
            try:
                os.makedirs(OUT, exist_ok=True)
                with open(path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                break
            except FileNotFoundError:
                if attempt == 4:
                    raise
                time.sleep(5)
        bar.update(1)
    bar.close()
    s = make_summary(rows, sp_name, mode, schema)
    save_summary(s)
    return s

t_all = time.time()
lines = []
for MODEL in RUN:                   # make_summary берёт ключ модели из глобальной MODEL
    mode, schema = BEST[MODEL]
    try:
        for sp_name in SPLITS:
            s = run_one(sp_name, mode, schema)
            if s:
                msg = f"{MODEL} {sp_name} {mode} {'схема' if schema else 'свободно'}: F1 {s['micro_f1_норм']}, ложные {s['ложные_на_null']}, {s['сек_среднее']} с"
                lines.append(msg)
                tg(msg)
    except Exception as e:
        tg(f"{MODEL}: ОШИБКА {type(e).__name__}: {str(e)[:300]}\nЗапусти ячейку ещё раз: готовые карточки и модели пропустятся.")
        if AUTO_OFF:                       # при ошибке тоже отключаемся: готовые карточки уже на Drive
            try:
                from google.colab import drive, runtime
                drive.flush_and_unmount()     # дождаться, пока все файлы уйдут на Google Drive
                runtime.unassign()
            except ImportError:
                pass
        raise
    finally:
        unload()
tg(f"heldout готово за {(time.time() - t_all) / 60:.0f} мин\n" + "\n".join(lines))

## H4: разрыв «heldout − real_val»
Нужны файлы `real_val__…jsonl` из основного прогона в той же папке `OUT`.

In [ ]:
# H4: разрыв F1 «heldout − real_val» для одной модели и одного режима, допуск 0,05 (06_pilot/thresholds.md).
# Наборы разные, поэтому bootstrap независимый: карточки каждого набора пересэмплируются отдельно.
def rows_of(sp_name, m, mode, schema):
    path = f"{OUT}/{sp_name}__{m}__{mode}__{'schema' if schema else 'free'}.jsonl"
    return list({r["id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}.values())

rng = np.random.default_rng(0)
B = 4000
res = []
for m, (mode, schema) in BEST.items():
    try:
        h, v = rows_of("heldout", m, mode, schema), rows_of("real_val", m, mode, schema)
    except FileNotFoundError:
        continue
    fh, fv = score(h, True)["micro_f1"], score(v, True)["micro_f1"]
    gaps = []
    for _ in range(B):
        hb = [h[i] for i in rng.integers(0, len(h), len(h))]
        vb = [v[i] for i in rng.integers(0, len(v), len(v))]
        gaps.append(score(hb, True)["micro_f1"] - score(vb, True)["micro_f1"])
    lo, hi = np.percentile(gaps, [2.5, 97.5])
    res.append({"модель": m, "режим": f"{mode}, {'схема' if schema else 'свободно'}", "F1 heldout": round(fh, 3),
                "F1 real_val": round(fv, 3), "разрыв": round(fh - fv, 3), "95% интервал": f"[{lo:.3f}; {hi:.3f}]",
                "H4 (разрыв ≤ 0,05)": "да" if fh - fv <= 0.05 else "нет"})
h4 = pd.DataFrame(res)
h4.to_csv(f"{OUT}/h4_heldout_gap.csv", index=False)
h4

In [ ]:
# Сначала сохранить, потом отключить. A100 тратит около 5,4 единицы в час, даже когда ничего не считает.
# Чтобы оставить сессию (например, посмотреть таблицы), поставь AUTO_OFF = False.
# 1) что лежит в папке результатов (файлы за последние сутки и сколько в них строк)
fresh = sorted(f for f in os.listdir(OUT) if os.path.isfile(f"{OUT}/{f}") and time.time() - os.path.getmtime(f"{OUT}/{f}") < 86400)
report = "\n".join(f"{f}: {sum(1 for _ in open(f'{OUT}/{f}', encoding='utf-8'))} строк" for f in fresh)
tg("Сохранено на Drive:\n" + (report or "новых файлов нет"))

if AUTO_OFF:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()     # 2) дождаться, пока все файлы уйдут на Google Drive
        tg("Данные на Drive, сессия Colab отключается")
        runtime.unassign()            # 3) отключить среду
    except ImportError:
        pass        # не Colab — отключать нечего
